In [4]:
import os
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch

# Cell: load and visualize embedding layers (PyTorch / state_dict-friendly)
# Save as a .py file or run as a notebook cell.

import matplotlib.pyplot as plt

try:
    import torch.nn as nn
except Exception:
    torch = None
    nn = None


def _to_numpy(tensor):
    if torch is not None and isinstance(tensor, torch.Tensor):
        return tensor.detach().cpu().numpy()
    return np.asarray(tensor)


def extract_embeddings_from_state_dict(state_dict):
    embeddings = {}
    for k, v in state_dict.items():
        arr = _to_numpy(v)
        if arr.ndim == 2:
            name = k
            # heuristic: prefer names indicating embedding
            if any(tok in name.lower() for tok in ("embed", "embedding", "emb")) or arr.shape[0] > 1:
                embeddings[name] = arr
    return embeddings


def extract_embeddings_from_module(module):
    embeddings = {}
    for name, mod in module.named_modules():
        if type(mod).__name__ in ("Embedding", "EmbeddingBag") or isinstance(mod, getattr(nn, "Embedding", ())) :
            w = getattr(mod, "weight", None)
            if w is not None:
                embeddings[name + ".weight"] = _to_numpy(w)
    # fallback: check state_dict
    try:
        sd = module.state_dict()
        embeddings.update(extract_embeddings_from_state_dict(sd))
    except Exception:
        pass
    return embeddings


def load_embeddings(path_or_obj):
    """
    Accepts:
      - path to a torch .pt/.pth file (state_dict or saved module)
      - a loaded torch.nn.Module instance
      - a state_dict (dict of tensors)
    Returns: dict name -> numpy array (num_embeddings, dim)
    """
    if isinstance(path_or_obj, dict):
        return extract_embeddings_from_state_dict(path_or_obj)
    if torch is not None and isinstance(path_or_obj, torch.nn.Module):
        return extract_embeddings_from_module(path_or_obj)
    if isinstance(path_or_obj, str):
        if torch is None:
            raise RuntimeError("PyTorch not available to load model file.")
        obj = torch.load(path_or_obj, map_location="cpu")
        if isinstance(obj, dict):
            # could be full checkpoint or state_dict
            # try common checkpoint keys
            if "state_dict" in obj and isinstance(obj["state_dict"], dict):
                return extract_embeddings_from_state_dict(obj["state_dict"])
            return extract_embeddings_from_state_dict(obj)
        if isinstance(obj, torch.nn.Module):
            return extract_embeddings_from_module(obj)
        # fallback: try state_dict via getattr
        try:
            sd = obj.state_dict()
            return extract_embeddings_from_state_dict(sd)
        except Exception:
            raise RuntimeError("Loaded object is not a module or state_dict.")
    raise TypeError("Unsupported input to load_embeddings.")


def plot_heatmap(weights, title=None, max_rows=200, figsize=(10, 6), cmap="viridis", save_path=None):
    arr = np.asarray(weights)
    if arr.shape[0] > max_rows:
        idx = np.linspace(0, arr.shape[0] - 1, max_rows, dtype=int)
        arr = arr[idx]
    plt.figure(figsize=figsize)
    sns.heatmap(arr, cmap=cmap, xticklabels=False, yticklabels=False)
    if title:
        plt.title(title)
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_pca_scatter(weights, title=None, annotate=10, figsize=(6, 5), save_path=None):
    arr = np.asarray(weights)
    pca = PCA(n_components=2)
    proj = pca.fit_transform(arr)
    plt.figure(figsize=figsize)
    plt.scatter(proj[:, 0], proj[:, 1], s=8, alpha=0.7)
    if annotate:
        for i in range(min(annotate, proj.shape[0])):
            plt.annotate(str(i), (proj[i, 0], proj[i, 1]), fontsize=8)
    if title:
        plt.title(title)
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_tsne_scatter(weights, title=None, perplexity=30, n_iter=1000, annotate=10, figsize=(6, 5), save_path=None):
    arr = np.asarray(weights)
    ts = TSNE(n_components=2, perplexity=min(perplexity, max(5, arr.shape[0] - 1)), n_iter=n_iter, init="pca", random_state=0)
    proj = ts.fit_transform(arr)
    plt.figure(figsize=figsize)
    plt.scatter(proj[:, 0], proj[:, 1], s=8, alpha=0.7)
    if annotate:
        for i in range(min(annotate, proj.shape[0])):
            plt.annotate(str(i), (proj[i, 0], proj[i, 1]), fontsize=8)
    if title:
        plt.title(title)
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close()


def visualize_embeddings(source, out_dir="embedding_plots", heatmap_max_rows=200):
    os.makedirs(out_dir, exist_ok=True)
    embeddings = load_embeddings(source)
    for name, w in embeddings.items():
        base = name.replace("/", "_").replace(" ", "_")
        hm_path = os.path.join(out_dir, f"{base}_heatmap.png")
        pca_path = os.path.join(out_dir, f"{base}_pca.png")
        tsne_path = os.path.join(out_dir, f"{base}_tsne.png")
        plot_heatmap(w, title=f"{name} heatmap", max_rows=heatmap_max_rows, save_path=hm_path)
        plot_pca_scatter(w, title=f"{name} PCA", save_path=pca_path)
        try:
            plot_tsne_scatter(w, title=f"{name} t-SNE", save_path=tsne_path)
        except Exception:
            pass
    return list(embeddings.keys())

In [5]:
visualize_embeddings("checkpoints\ppo\ppo_update_120_1459200_eval_-0.255.pt")

C:\Users\tyler\AppData\Local\Temp\ipykernel_16120\2377219321.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(path_or_obj, map_location="cpu")


[]